
# Notebook 3 — Qualidade dos dados do GWOSC

**Estudo de caso: GW150914 · LIGO Hanford (H1) · Campanha O1**

### Objetivos

- Compreender o significado das informações de qualidade dos dados (*Data Quality — DQ*) disponibilizadas pelo GWOSC.
- Interpretar as máscaras de bits (*DQmask*) e identificar as categorias de qualidade dos dados.
- Utilizar operações binárias para verificar quais condições de qualidade são satisfeitas.
- Identificar e visualizar os segmentos de dados adequados para diferentes tipos de análise.

> **Como trabalhar:** Execute as células na ordem. Nas pausas **Observe**, interprete os resultados e compare sua saída com a projetada em sala.

**Origem:** Material adaptado de [Introduction to GWOSC Data](https://github.com/gwosc-tutorial/introduction_gwosc_data), sob licença GPL-3.0 ou posterior.

**Adaptação didática:** Ronai Lisbôa — UFRN.


# 1. Importar as bibliotecas

**Google Colab:**

* Execute a célula seguinte para baixar automaticamente o arquivo HDF5 do GWOSC, caso ainda não esteja disponível.

* Em ambiente local, o download será dispensado quando o arquivo já estiver na pasta de trabalho.

In [ ]:
# requests é uma biblioteca utilizada para realizar requisições pela internet.
# Neste curso, ela será usada para consultar a API do GWOSC e baixar arquivos.
import requests

# Exibe a versão instalada da biblioteca requests.
print("Versão do requests:", requests.__version__)


# GWpy é uma biblioteca Python desenvolvida para acessar, representar,
# visualizar e analisar dados de detectores de ondas gravitacionais.
import gwpy

# TimeSeries é uma classe do GWpy que representa dados medidos ao longo do tempo,
# como a série temporal de strain dos detectores LIGO e Virgo.
from gwpy.timeseries import TimeSeries

# Exibe a versão instalada da biblioteca GWpy.
print("Versão do GWpy:", gwpy.__version__)

# Path manipula caminhos, pastas e arquivos.
from pathlib import Path

# h5py lê arquivos HDF5; Matplotlib cria gráficos; NumPy realiza cálculos numéricos.
import h5py
import matplotlib.pyplot as plt
import numpy as np

# Define o tamanho padrão das figuras e das fontes.
plt.rcParams.update({"figure.figsize": (11, 4), "font.size": 14})

In [ ]:
# Localizar o arquivo HDF5 na pasta atual.
pasta_dados = Path.cwd()

filename = pasta_dados / "H-H1_LOSC_4_V1-1126256640-4096.hdf5"

if not filename.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado em: {filename}"
    )

print(f"Arquivo localizado em: {filename}")

## 1.1 Construir o gráfico do strain x tempo

In [ ]:
#-----------------------------------------------------------------------------
# Construir o gráfico do strain x tempo com o móudlo pyplot (aliás plt)
#-----------------------------------------------------------------------------
import h5py

# h5py.File: classe que cria um objeto e o referencia à variável dataFile
dataFile = h5py.File(filename, 'r')

# acessa o grupo strain e o dataset Strain e o referencia a variável strain.
strain = dataFile['strain']['Strain']

# strain.attrs: recupera o intervalo entre amostras e o referencia à variável dt.
dt = strain.attrs['Xspacing']

# operações numéricas
duracao = 4096
N = int(duracao / dt)

# Selecionar as primeiras N amostras do arquivo
seg_strain = strain[:N]

# Constrói o eixo do tempo relativo ao início do arquivo
rel_time = np.arange(len(seg_strain)) * dt

plt.figure(figsize=(10, 3))
plt.plot(rel_time, seg_strain)

plt.title('Strain contendo o sinal GW150914')
plt.xlabel('Tempo (s)')
plt.ylabel('Strain')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.axvline(2822.4, color='red', linestyle='--', label='GW150914')
plt.legend(loc='upper right')
plt.show()

Sinal de strain registrado pelo detector LIGO Hanford (H1) durante 4096 segundos da campanha O1.

<div align="center">

<img src="https://raw.githubusercontent.com/Ronai/gwd/c765c02a84cf164c3bbeceefd79ff759ea4cc11f/introduction_gwosc_data/img/ligo%20-%20strain%20GW150914.png" width="750">

</div>

*   Interpretando os eixos
*   O que representam as diferentes partes do gráfico?
*   O que esse gráfico nos ensina sobre qualidade dos dados?

## 1.2 Observando o strain próximo ao evento GW150914.

Anteriormente, representamos os 4096 segundos de dados do detector LIGO Hanford. Agora vamos ampliar a região próxima ao evento GW150914, ocorrido aproximadamente 2822,4 segundos após o início do arquivo.

Agora, a série temporal a seguir apresenta um intervalo de quatro segundos, centrado no evento.

> **Observe:** Mesmo ampliando a escala temporal, conseguimos distinguir claramente o sinal gravitacional das flutuações de ruído?
>
> A linha vermelha indica o instante conhecido do evento, mas não significa que sua forma de onda seja diretamente identificável no *strain* bruto.

In [ ]:
# ────── Visualizar o strain ao redor do evento GW150914 ──────

# Instante do evento em relação ao início do arquivo
tempo_evento = 2822.4

# Selecionar 2 segundos antes e 2 segundos depois do evento
inicio = int((tempo_evento - 2) / dt)
fim = int((tempo_evento + 2) / dt)

seg_strain = strain[inicio:fim]

# Construir o eixo temporal relativo ao início do arquivo
rel_time = np.arange(inicio, fim) * dt

plt.figure(figsize=(10, 3))
plt.plot(rel_time, seg_strain)

plt.title('Strain ao redor do evento GW150914')
plt.xlabel('Tempo desde o início do arquivo (s)')
plt.ylabel('Strain')
plt.grid(True, alpha=0.3)

# Marcar o instante do evento
plt.axvline(tempo_evento, color='red', linestyle='--', label='GW150914')
plt.legend(loc='upper right')

plt.tight_layout()
plt.show()


# 2. O que é qualidade de dados?

Além da principal saída de dados dos detectores LIGO (o canal de *strain*), existem centenas de outros canais de dados registrados para monitorar o estado tanto dos instrumentos quanto do ambiente externo.

Os detectores LIGO registram continuamente informações sobre o funcionamento dos instrumentos e as condições ambientais. Entretanto, nem todos os intervalos de dados são adequados para a busca de ondas gravitacionais.

Vibrações sísmicas, problemas de controle e outras perturbações instrumentais podem contaminar o sinal de *strain*. Por isso, antes de realizar uma análise, precisamos identificar os intervalos em que os dados apresentam condições adequadas.



### 2.1 Principais subsistemas de controle do interferômetro

<div align="center">

<img src="https://raw.githubusercontent.com/Ronai/gwd/0ebbc880ac71858efb0ef3c9ed4b948000704826/introduction_gwosc_data/img/ligo%20-%20adv%20configuration%20O3%20run.png" width="750">

<p style="font-size:12px; color:gray;">
Fonte: Virgo Collaboration,
<a href="https://arxiv.org/html/2210.15633v2">arXiv:2210.15633</a>,
Fig. X (2023). CC BY 4.0.
</p>

</div>


- **PSL (*<a href="https://www.aei.mpg.de/38499/advanced-ligo">Pre-Stabilized Laser</a>*):** sistema responsável pela pré-estabilização do laser, reduzindo flutuações de frequência e potência antes que a luz entre no interferômetro.

- **SSFS (*<a href="https://www.sciencedirect.com/science/article/pii/S0141635914001561">Second Stage Frequency Stabilization</a>*):** segundo estágio de estabilização da frequência do laser, que utiliza uma referência óptica para reduzir ainda mais suas flutuações.

- **PRCL (*Power Recycling Cavity Length*):** controla o comprimento efetivo da cavidade de reciclagem de potência, permitindo manter a condição de ressonância óptica e aumentar a potência luminosa no interferômetro.

- **MICH (*Michelson*):** controla a diferença entre os comprimentos dos dois braços curtos do interferômetro de Michelson, mantendo a condição de interferência desejada.

- **DARM (*Differential Arm Length*):** controla a diferença entre os comprimentos dos dois braços principais do interferômetro. É o grau de liberdade diretamente associado à medição das ondas gravitacionais e à reconstrução do sinal de *strain*.


# 3. Como interpretar a qualidade dos dados?

Alguns canais auxiliares são utilizados para criar indicadores de qualidade dos dados, assinalando momentos em que os dados de *strain* podem estar contaminados por artefatos instrumentais.

Para essa finalidade, o GWOSC disponibiliza indicadores de qualidade dos dados, denominados **Data Quality Flags** (DQ flags).

**Para saber mais:** consulte a [documentação de qualidade dos dados da campanha O1](https://gwosc.org/archive/dataset/O1/) e o artigo [Characterization of transient noise in Advanced LIGO relevant to gravitational wave signal GW150914](https://arxiv.org/abs/1602.03844).

## 3.1 Categorias de qualidade dos dados

As informações de qualidade dos dados são armazenadas em arquivos de dados GWOSC como uma série temporal de 1 Hz para cada categoria.

Observe que esta é uma taxa de amostragem diferente da taxa de 4096 Hz usada para armazenar dados de strain.

As flags são organizadas em categorias de acordo com a gravidade dos problemas identificados.

| Categoria | Significado | Interpretação |
|---|---|---|
| **DATA** | Disponibilidade dos dados | Indica se os dados estão disponíveis no GWOSC. |
| **CAT1** | Problemas críticos | Identifica falhas graves no funcionamento ou na configuração do detector. |
| **CAT2** | Contaminações conhecidas | Identifica intervalos com acoplamentos físicos conhecidos ao canal de *strain*, como algumas perturbações sísmicas. |
| **CAT3** | Contaminações estatísticas | Identifica intervalos com evidências estatísticas de contaminação, mas cujo mecanismo físico não é completamente compreendido. |

**Importante:** uma categoria de número menor corresponde, em geral, a um problema de maior gravidade.

As categorias são cumulativas: se um intervalo falha em `CAT2`, por exemplo, também falha em `CAT3` para o mesmo tipo de análise.

<div style="background-color:#FFF4D6; border-left:5px solid #D97706; border-radius:6px; padding:14px 18px; margin:12px 0; color:#573A11;">
  <strong style="font-size:1.05em;">⚠️ Atenção — qualidade dos dados</strong>
  <p style="margin:8px 0 0;">Compreender a qualidade dos dados é muito importante ao trabalhar com dados do GWOSC. Antes de realizar qualquer análise, utilize este tutorial e a <a href="https://gwosc.org/O1">documentação dos dados</a> para identificar os intervalos de tempo adequados para a sua análise.</p>
</div>

In [ ]:
# ----------------------------------------------------------------------------
# Ler o arquivo e listar os canais de qualidade dos dados ──────
# ----------------------------------------------------------------------------

#  dqInfo:      contém as informações das flags de qualidade.
#  bitnameList e descriptionList armazenam seus nomes e descrições.
#  o método .decode() converte os textos armazenados como byte.

import h5py

# h5py.File: classe que abre o arquivo HDF5 no modo leitura ('r').
# dataFile : variável que referencia o objeto criado.
dataFile = h5py.File(filename, 'r')

# Acessa o grupo 'meta' e o dataset 'GPSstart'.
# [()]: recupera o valor armazenado no dataset.
gpsStart = dataFile['meta']['GPSstart'][()]

# Acessa os grupos 'quality' e 'simple'.
# dqInfo: variável que referencia o grupo de qualidade dos dados.
dqInfo = dataFile['quality']['simple']

# Recupera o array com os nomes das categorias de qualidade.
bitnameList = dqInfo['DQShortnames'][()]

# Recupera o array com as descrições das categorias.
descriptionList = dqInfo['DQDescriptions'][()]

# len(): função que retorna o número de elementos do array.
nbits = len(bitnameList)

# for: estrutura de repetição.
# range(): função que gera os índices de 0 até nbits - 1.
for bit in range(nbits):
    print(f"Canal #{bit} ({bitnameList[bit].decode()}): {descriptionList[bit].decode()}")


## 3.2 A qualidade depende do tipo de análise

Diferentes sinais de ondas gravitacionais podem ser afetados de maneiras distintas pelo mesmo ruído instrumental.

Por isso, as categorias são definidas para diferentes tipos de busca, incluindo:

- **CBC (*Compact Binary Coalescence*):** sinais produzidos pela coalescência de sistemas binários compactos, como dois buracos negros.
- **BURST:** sinais gravitacionais transientes cuja forma de onda não é necessariamente conhecida.
- **CW (*Continuous Waves*):** sinais aproximadamente contínuos, como aqueles que podem ser emitidos por estrelas de nêutrons em rotação.
- **Stochastic:** busca por um fundo estocástico de ondas gravitacionais.

Um intervalo pode satisfazer `CBC_CAT2` e, ao mesmo tempo, não satisfazer `BURST_CAT2`.

Isso ocorre porque os critérios de qualidade são definidos separadamente para cada tipo de busca.


## 3.3 Como funcionam as flags?

As flags de qualidade dos dados indicam se um determinado intervalo de observação satisfaz condições estabelecidas para a análise de ondas gravitacionais.

As flags são organizadas em categorias:

| Categoria | Significado |
|---|---|
| **DATA** | Indica a disponibilidade dos dados. |
| **CAT1** | Identifica problemas instrumentais críticos. |
| **CAT2** | Identifica contaminações instrumentais com mecanismos físicos conhecidos. |
| **CAT3** | Identifica contaminações estatísticas cujo mecanismo físico não é completamente compreendido. |

As categorias são **cumulativas**: um intervalo que não satisfaz `CAT2`, por exemplo, também não satisfaz `CAT3` para o mesmo tipo de análise.

Além disso, os critérios são definidos separadamente para diferentes buscas, como coalescência de binárias compactas (**CBC**) e sinais gravitacionais transientes (**BURST**).

> **⚠️ Atenção:** Os intervalos que não satisfazem `DATA` ou `CAT1` não devem ser utilizados nas buscas astrofísicas. Para as análises da campanha O1, os critérios de seleção devem seguir as recomendações específicas do tipo de busca.


## 3.4 Identificando as flags no arquivo HDF5

Na campanha O1, o GWOSC disponibiliza **7 flags de qualidade dos dados e 5 indicadores de injeção**, cada um representado por uma série temporal de 1 Hz.

As sete flags de qualidade estão distribuídas entre a disponibilidade dos dados (`DATA`) e as categorias utilizadas nas buscas CBC e BURST.

Vamos identificar seus nomes e descrições diretamente no arquivo HDF5.

```python
Canal #0 (DATA): data present
Canal #1 (CBC_CAT1): passes the cbc CAT1 test
Canal #2 (CBC_CAT2): passes cbc CAT2 test
Canal #3 (CBC_CAT3): passes cbc CAT3 test
Canal #4 (BURST_CAT1): passes burst CAT1 test
Canal #5 (BURST_CAT2): passes burst CAT2 test
Canal #6 (BURST_CAT3): passes burst CAT3 test
```


> **Observe:** O canal `DATA` é comum às análises, enquanto CBC e BURST possuem três categorias próprias de qualidade.
>
> Cada canal está associado a uma posição (*bit*) na máscara `DQmask`, que estudaremos a seguir.


# 4. Interpretando a máscara de qualidade dos dados (`DQmask`)

No arquivo HDF5, as informações de qualidade dos dados são armazenadas em um array denominado `DQmask`, contendo 4096 elementos, um para cada segundo de observação.

Cada elemento é um número inteiro que reúne as informações de sete condições de qualidade dos dados. Essas condições são representadas pelos sete bits de sua representação binária.

Cada posição binária, chamada bit, corresponde a uma condição de qualidade:
- 1: a condição de qualidade é satisfeita.
- 0: a condição de qualidade não é satisfeita.

Assim, um número inteiro pode representar simultaneamente o estado das sete condições.

<div align="center">

<img src="https://raw.githubusercontent.com/Ronai/gwd/596efff96e2f4aa4e7229c6d041353e0e3471655/introduction_gwosc_data/img/ligo%20-%20hdf5-dqmask.png" width="750">



</div>

In [ ]:
# ---------------------------------------------------
# Extrair a máscara de qualidade dos dados (DQmask)
# ---------------------------------------------------

# Acessa o dataset DQmask e extrai o array de valores.
dqmask = dqInfo['DQmask'][()]

# Seleciona o elemento de índice 4080 (um segundo de observação).
value = dqmask[4080]

# Exibe o valor em decimal.
# .format(): método que insere o valor na string.
print("Value in decimal: {0}".format(value))

# Exibe o mesmo valor em binário, com 7 bits.
# #09b: inclui o prefixo 0b e completa com zeros à esquerda.
print("Same value in binary (with 7 bits): {0:#09b}".format(value))

No exemplo, selecionamos o elemento `dqmask[4080]` e encontramos o valor decimal `127`, cuja representação binária é `1111111`.

Isso significa que todas as sete condições de qualidade são satisfeitas naquele segundo.


<div align="center">

<img src="https://raw.githubusercontent.com/Ronai/gwd/f34ce6ecacd8c6d5dc2bf8a332315045737e1f3c/introduction_gwosc_data/img/ligo%20-%20dqmask.png" width="750">

</div>


Se alguma condição não fosse satisfeita, o bit correspondente teria valor zero, produzindo outro número inteiro.

> **Observe:** O número decimal 127 não é uma medida da intensidade ou da qualidade do sinal. Ele é apenas uma forma compacta de registrar o resultado de sete testes de qualidade.

Esse binário quer dizer que:
- Bit 0: 1 -> DATA (Aprovado: os dados estão presentes e o detector estava ligado)
- Bit 1: 1 -> CBC_CAT1 (Aprovado: passou no veto gravíssimo de CBC)
- Bit 2: 1 -> CBC_CAT2 (Aprovado: passou no veto intermediário de CBC)
- Bit 3: 1 -> CBC_CAT3 (Aprovado: passou no veto brando de CBC)
- Bit 4: 1 -> BURST_CAT1 (Aprovado: passou no veto gravíssimo de BURST)
- Bit 5: 1 -> BURST_CAT2 (Aprovado: passou no veto intermediário de BURST)
- Bit 6: 1 -> BURST_CAT3 (Aprovado: passou no veto brando de BURST)

## 4.1 Verificando outro segundo.

 Vamos experimentar outro segundo. Por exemplo, referente a entrada de número 12 da flag DQ qualy no arquivo hdf5.

In [ ]:
# ---------------------------------------------------
#  Extrair a máscara de qualidade dos dados (DQmask)
# ---------------------------------------------------

dqmask = dqInfo['DQmask'][()]

value = dqmask[12]

print("Value in decimal: {0}".format(value))
print("Same value in binary (with 7 bits): {0:#09b}".format(value))

```pyhton
Value in decimal: 63
Same value in binary (with 7 bits): 0b0111111
```

Assim, o valor decimal `63` é representado em binário como `0 111 111` (para maior clareza, apresentamos os bits em grupos de 3 e descartamos o prefixo `0b` utilizado no Python).

Essa representação é chamada de máscara de bits.

Essa é uma maneira compacta de armazenar uma grande quantidade de informações, mas, para trabalhar com a qualidade dos dados, precisaremos colocá-las em um formato mais conveniente.

Vamos tentar desmembrar algumas dessas informações sobre a qualidade dos dados.


<div align="center">

<img src="https://raw.githubusercontent.com/Ronai/gwd/bf82ac0b700a4789055fe1fd09a542a1cd2fe7ca/introduction_gwosc_data/img/ligo%20-%20dqmask%20codigo.png" width="750">

</div>

Um dígito "1" em uma posição específica significa que a flag correspondente está ativa, indicando que os dados são "bons" naquele nível de categoria; já um dígito "0" significa que a flag está desativada, indicando que os dados são "ruins" naquele nível de categoria.

- os dados do detector estão disponíveis em `DATA` (porque o bit na posição 0 é 1)
- os dados atendem aos critérios `CBC_CAT1`, `CBC_CAT2`, `CBC_CAT3`, `BURST_CAT1` e `BURST_CAT2` (já que os bits nas posições 1 a 5 são 1), mas não cumprem a a condição de qualidade dos dados `BURST_CAT3` (bit 6).

## 4.2 Como verificar uma categoria específica de qualidade?


A máscara `DQmask` reúne sete condições de qualidade em um único número inteiro. Entretanto, muitas vezes queremos verificar apenas uma dessas condições.

Para isso, utilizamos dois operadores do Python:

- **`>>` (deslocamento à direita):** desloca os bits de um número para a direita, colocando o bit que desejamos examinar na posição mais à direita.
- **`&` (AND binário):** permite isolar esse bit, utilizando a operação `& 1`.

Considere a máscara:

`DQmask = 63 = 0b0111111`

Para verificar o bit 3 (`CBC_CAT3`), executamos:

```python
resultado = (63 >> 3) & 1
```

Primeiro, deslocamos todos os bits três posições para a direita:

`0111111 >> 3 = 0000111`


<div align="center">

<img src="https://raw.githubusercontent.com/Ronai/gwd/afc7929b87b0c5983f9b8106ff568f012e77e8e1/introduction_gwosc_data/img/ligo%20-%20desloque%20bit.png" width="750">

</div>


Em seguida, aplicamos a operação AND com 1:

`0000111 & 0000001 = 0000001`

- A função do AND é definir se o resultado do bit 0 é válido (1) ou  não (0).
- Ele faz essa comparação com o bit 0 apenas, por isso, faz-se necessário fazer o deslocamento.
- Na prática ele faz uma comparação lógica com 1. Se 1 AND 1 resulta em 1, se 0 AND 1 resulta em 0.

Portanto, o resultado é **1**, indicando que a condição de qualidade representada pelo bit 3 é satisfeita.

De maneira geral, podemos consultar qualquer bit utilizando:

```python
resultado = (int(value) >> bit) & 1
```

> **Observe:** o deslocamento `>>` não verifica sozinho se uma condição foi satisfeita. Ele apenas reposiciona os bits. A operação `& 1` é responsável por isolar o valor do bit desejado.


## 4.3 Selecionando categorias específicas de qualidade

Anteriormente, utilizamos os operadores `>>` e `& 1` para verificar o valor de um único bit da máscara `DQmask`.

Podemos aplicar o mesmo procedimento a **todos os 4096 elementos do array**, identificando quais segundos de observação satisfazem uma determinada condição de qualidade.

Como exemplo, vamos selecionar duas categorias:

- **`DATA` (bit 0):** indica se os dados estão disponíveis.
- **`BURST_CAT3` (bit 6):** indica se os dados satisfazem os critérios da Categoria 3 para buscas de sinais gravitacionais transientes (*bursts*).

Para cada categoria, utilizamos o deslocamento `>>` para posicionar o bit desejado à direita e a operação `& 1` para extrair seu valor.

O resultado será um novo array contendo apenas valores **0 e 1**, com um elemento para cada segundo de observação.


In [ ]:
# -----------------------------------------------------------------
# Selecionar as categorias DATA e BURST_CAT3
# -----------------------------------------------------------------

# Posição do bit correspondente à categoria DATA.
data_channel = 0

# Extrai o bit 0 de cada um dos 4096 elementos de dqmask.
# O resultado é um novo array com valores 0 ou 1.
data_mask = (dqmask >> data_channel) & 1

# Posição do bit correspondente à categoria BURST_CAT3.
burst_cat3_channel = 6

# Extrai o bit 6 de cada um dos 4096 elementos de dqmask.
# O resultado é outro array com valores 0 ou 1.
burst_cat3_mask = (dqmask >> burst_cat3_channel) & 1



# ────── Comparar DQmask com os arrays resultantes ──────

print(f"{'Índice':<8} {'Decimal':<10} {'Binário':<12} {'data_mask':<12} {'burst_cat3_mask'}")

for i in [0, 1, 2, 4093, 4094, 4095]:
    print(f"{i:<8} {dqmask[i]:<10} {dqmask[i]:07b}      {data_mask[i]:<12} {burst_cat3_mask[i]}")


```python
Índice   Decimal    Binário      data_mask    burst_cat3_mask
0        31         0011111      1            0
1        19         0010011      1            0
2        31         0011111      1            0
4093     127        1111111      1            1
4094     127        1111111      1            1
4095     127        1111111      1            1
```

Agora temos dois arrays:

- **`data_mask`:** contém 1 nos segundos que satisfazem a condição `DATA` e 0 nos demais.
- **`burst_cat3_mask`:** contém 1 nos segundos que satisfazem a condição `BURST_CAT3` e 0 nos demais.

Observe que não estamos mais analisando apenas um segundo, como fizemos com `dqmask[12]`. As operações são realizadas sobre **todos os elementos do array simultaneamente**.

Dessa forma, podemos identificar os intervalos de tempo que satisfazem cada condição de qualidade.


## 4.4 Selecionando intervalos que satisfazem ambas as condições

Em uma análise de ondas gravitacionais, geralmente precisamos selecionar intervalos que satisfaçam **mais de uma condição de qualidade ao mesmo tempo**.

Por exemplo, queremos identificar os segundos em que:

1. Os dados estão disponíveis (`DATA = 1`).
2. Os dados satisfazem a Categoria 3 para buscas de *bursts* (`BURST_CAT3 = 1`).

Para isso, utilizamos o operador lógico **AND (`&`)**, aplicado elemento a elemento aos dois arrays.

| DATA | BURST_CAT3 | Resultado |
|:---:|:---:|:---:|
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | **1** |

O resultado será **1 somente quando as duas condições forem satisfeitas**.

Assim, construímos uma nova máscara que identifica os segundos de observação adequados segundo os dois critérios escolhidos.


In [ ]:
#----------------------------------------------------------------------------
# Combinar as categorias com uma operação AND
#----------------------------------------------------------------------------

# Operação AND (&) elemento a elemento entre os dois arrays.
# O resultado será 1 somente quando ambas as flags forem 1.
goodData_mask_1hz = data_mask & burst_cat3_mask

#----------------------------------------------------------------------------
# Verificar o resultado em um segundo de observação
#----------------------------------------------------------------------------

# Seleciona o índice do segundo que queremos analisar.
segundo = 12

# Exibe os valores das flags e o resultado da operação AND.
print(f"{'Segundo':<10} {'DATA':<10} {'BURST_CAT3':<15} {'AND'}")
print(f"{segundo:<10} {data_mask[segundo]:<10} {burst_cat3_mask[segundo]:<15} {goodData_mask_1hz[segundo]}")



## 4.5. Visualizando a combinação das categorias de qualidade

Para verificar o resultado da operação AND, vamos representar graficamente as três máscaras de qualidade:

- **`DATA`:** indica os segundos em que os dados estão disponíveis.
- **`BURST_CAT3`:** indica os segundos que satisfazem os critérios de qualidade da Categoria 2 para buscas de *bursts*.
- **`goodData_mask_1hz`:** identifica os segundos em que **ambas as condições são satisfeitas simultaneamente**.

Em cada gráfico, o valor **1** indica que a condição é satisfeita, enquanto o valor **0** indica que ela não é satisfeita.

Compare os três gráficos e observe que a máscara resultante será igual a 1 somente nos intervalos em que `DATA` e `BURST_CAT3` também são iguais a 1.


In [ ]:
#----------------------------------------------------------------------------
# Comparar as máscaras de qualidade dos dados
#----------------------------------------------------------------------------

fig, (ax0, ax1, ax2) = plt.subplots(
    3, sharex=True, sharey=True, figsize=(11, 6)
)

ax0.step(range(len(goodData_mask_1hz)), goodData_mask_1hz, where='post')
ax0.set_title('Dados que satisfazem ambas as condições (AND)')

ax1.step(range(len(burst_cat3_mask)), burst_cat3_mask, where='post')
ax1.set_title('BURST_CAT3')

ax2.step(range(len(data_mask)), data_mask, where='post')
ax2.set_title('DATA')

for ax in (ax0, ax1, ax2):
    ax.set_yticks([0, 1])
    ax.set_ylabel('Flag')
    ax.grid(True, alpha=0.3)

ax2.set_xlim(0, len(dqmask))
ax2.set_ylim(-0.2, 1.2)
ax2.set_xlabel('Tempo desde o início do arquivo (s)')

plt.tight_layout()
plt.show()



> **Observe:** A condição `DATA` é satisfeita durante os 4096 segundos analisados. Entretanto, a máscara `BURST_CAT3` apresenta diversos intervalos em que seu valor é zero.
>
> Esses intervalos correspondem a períodos que não satisfazem os critérios de qualidade da Categoria 3 para buscas de sinais gravitacionais transientes (*bursts*).
>
> Como a máscara `DATA` é sempre igual a 1, a operação AND reproduz exatamente a máscara `BURST_CAT3`.
>
> **Questão:**

>Por que alguns intervalos não satisfazem `BURST_CAT3`, mesmo quando os dados estão disponíveis (`DATA = 1`)? O que isso nos diz sobre a diferença entre disponibilidade e qualidade dos dados?



## 4.6. Criando a Lista de Segmentos Limpos (** **segList** **) e Plotando o Strain por Fatias

Como temos trechos de dados inválidos ou com ruídos, nós não precisaremos analisar todos os 4096 dados daquele bloco, então vamos criar uma lista de segmentos limpos.

Além disso, para não passar os nossos arrays como parâmetros, precisamos passar o intervalo de tempo para ficar mais simples.

Para extrair os segmentos contínuos usaremos funções do NumPy.

In [ ]:
#----------------------------------------------------------------------------
# Identificar segmentos contínuos de dados válidos
#----------------------------------------------------------------------------

# np.zeros(): cria um array de zeros com o mesmo tamanho de goodData_mask_1hz.
dummy = np.zeros(goodData_mask_1hz.shape)

# np.logical_not()    : troca 1 por False e 0 por True.
# np.ma.masked_array(): cria um array mascarado,
# ocultando os elementos onde goodData_mask_1hz vale 0.
masked_dummy = np.ma.masked_array(dummy, np.logical_not(goodData_mask_1hz))

# np.ma.flatnotmasked_contiguous(): identifica os blocos contínuos não mascarados,
# isto é, os intervalos em que goodData_mask_1hz = 1.
segments = np.ma.flatnotmasked_contiguous(masked_dummy)

# List comprehension: converte cada segmento em uma tupla
# (GPS inicial, GPS final), somando gpsStart aos índices.
segList = [(int(seg.start + gpsStart), int(seg.stop + gpsStart)) for seg in segments]

# for: percorre a lista de segmentos e imprime cada intervalo.
for seg in segList:
    print(seg)



## 4.7 Identificando os segmentos válidos

Os intervalos apresentados correspondem aos segmentos contínuos de observação que satisfazem simultaneamente as condições `DATA` e `BURST_CAT3`.

Cada par `(GPS inicial, GPS final)` identifica um intervalo de tempo em que ambas as flags permanecem iguais a 1.

Esses segmentos são considerados **válidos segundo os critérios de qualidade selecionados**, mas não necessariamente livres de ruído instrumental.

> **Observe:** Compare as durações dos segmentos. Por que alguns intervalos são muito curtos, enquanto outros permanecem válidos por centenas de segundos?


## 4.8 Visualizando o sinal de *strain* nos segmentos válidos

Agora podemos traçar os primeiros segundos de cada segmento.

Após identificar os intervalos que satisfazem os critérios de qualidade, podemos examinar o sinal de *strain* em cada um desses segmentos.

Para isso, utilizamos os tempos GPS inicial e final armazenados em `segList` e extraímos as amostras correspondentes do arquivo HDF5.

Como cada segmento pode conter milhares de amostras, vamos representar apenas as **primeiras 10.000 amostras** de cada intervalo.

> **Observe:** Compare os gráficos obtidos. O sinal de *strain* apresenta características semelhantes em todos os segmentos? A seleção por flags de qualidade garante que o sinal esteja completamente livre de ruído?


In [ ]:
#----------------------------------------------------------------------------
# Visualizar os primeiros valores de strain em cada segmento
#----------------------------------------------------------------------------

N = 10000

strain = dataFile['strain']['Strain']
dt = strain.attrs['Xspacing']

for i, (begin, end) in enumerate(segList, start=1):

    # Converter os tempos GPS em índices do array
    index_begin = int((begin - gpsStart) / dt)
    index_end = int((end - gpsStart) / dt)

    # Extrair no máximo N amostras do segmento
    seg_strain = strain[index_begin:min(index_begin + N, index_end)]

    # Tempo relativo ao início do segmento
    rel_time = np.arange(len(seg_strain)) * dt

    plt.figure(figsize=(10, 3))
    plt.plot(rel_time, seg_strain)

    plt.title(f'Região aceita {i} — primeiros 2,44 s (GPS {begin})')
    plt.xlabel('Tempo desde o início do segmento (s)')
    plt.ylabel('Strain')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
#----------------------------------------------------------------------------
# Visualizar o strain completo e as flags de qualidade
#----------------------------------------------------------------------------

# Selecionar os primeiros N segundos
duracao = 4096
N = int(duracao / dt)

tempo = np.arange(N) * dt
sinal = strain[:N]

# Máscara de qualidade: 1 valor por segundo
mascara = goodData_mask_1hz[:duracao]

# Expandir a máscara de 1 Hz para a frequência do strain
amostras_por_segundo = int(round(1 / dt))
mascara_strain = np.repeat(mascara, amostras_por_segundo)[:N]

# Identificar amostras que não satisfazem os critérios
sinal_rejeitado = np.where(mascara_strain == 0, sinal, np.nan)

fig, (ax0, ax1) = plt.subplots(
    2, figsize=(11, 6), sharex=True, sharey=True
)

# Sinal original
ax0.plot(tempo, sinal)
ax1.set_title('Strain original com intervalos rejeitados destacados em vermelho')
ax0.set_ylabel('Strain')

# Mesmo sinal com intervalos rejeitados destacados
ax1.plot(tempo, sinal, color='steelblue', alpha=0.5)
ax1.plot(tempo, sinal_rejeitado, color='red')
ax1.set_title('Strain com intervalos rejeitados por DATA e BURST_CAT3')
ax1.set_xlabel('Tempo desde o início do arquivo (s)')
ax1.set_ylabel('Strain')

for ax in (ax0, ax1):
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



> **Observe:** O gráfico superior apresenta os 4096 segundos do sinal de *strain* original, incluindo dois transientes de grande amplitude, próximos ao início do arquivo e a 2950 segundos.
>
> No gráfico inferior, o mesmo sinal é apresentado com os intervalos que **não satisfazem simultaneamente `DATA` e `BURST_CAT3` destacados em vermelho**. As regiões azuis correspondem aos intervalos aceitos.
>
> Note que os dois transientes de grande amplitude coincidem com intervalos rejeitados. Entretanto, existem outros intervalos rejeitados nos quais não observamos perturbações tão evidentes. Isso ocorre porque as flags consideram diferentes informações sobre o funcionamento do detector e suas condições ambientais, e não apenas a amplitude do *strain*.
>
> **Questão:** Por que alguns intervalos são rejeitados mesmo quando não apresentam transientes claramente visíveis? Um intervalo aceito está necessariamente livre de ruído?
>
> **Conclusão:** As flags de qualidade não modificam o sinal de *strain*. Elas permitem identificar os intervalos que satisfazem determinados critérios e selecionar os dados mais adequados para cada tipo de análise.


In [ ]:

# ────── Comparar a amplitude do strain com a qualidade dos dados ──────

# Número de amostras de strain por segundo.
fs = int(round(1 / dt))

# Calcular o RMS do strain a cada segundo.
Nseg = len(goodData_mask_1hz)
sinal = strain[:Nseg * fs]

# reshape(): organiza as amostras em Nseg linhas, cada uma com fs amostras.
# np.mean(): calcula a média dos quadrados em cada segundo.
# np.sqrt(): calcula a raiz quadrada, obtendo o RMS.
strain_rms = np.sqrt(
    np.mean(sinal.reshape(Nseg, fs)**2, axis=1)
)

# Tempo relativo ao início do arquivo.
tempo = np.arange(Nseg)

# Instante GPS do evento GW150914.
gps_evento = 1126259462.4
tempo_evento = gps_evento - gpsStart

# Cria dois gráficos com o mesmo eixo temporal.
fig, (ax0, ax1) = plt.subplots(
    2, figsize=(12, 6), sharex=True,
    gridspec_kw={'height_ratios': [3, 1]}
)

# ────── Gráfico superior: amplitude RMS do strain ──────

ax0.plot(tempo, strain_rms)
ax0.axvline(tempo_evento, color='darkred', linestyle='--',
            linewidth=2, label='GW150914')

ax0.set_title('Amplitude RMS do strain ao longo do arquivo')
ax0.set_ylabel('Strain RMS')
ax0.grid(True, alpha=0.3)
ax0.legend(loc='upper right')

# ────── Gráfico inferior: intervalos de qualidade ──────

# Define os limites de cada intervalo de 1 segundo.
limites = np.arange(Nseg + 1)

# Destaca em azul os intervalos aceitos (AND = 1).
ax1.stairs(
    goodData_mask_1hz,
    edges=limites,
    baseline=0,
    fill=True,
    color='steelblue',
    label='Aceito'
)

# Destaca em vermelho os intervalos rejeitados (AND = 0).
ax1.stairs(
    1 - goodData_mask_1hz,
    edges=limites,
    baseline=0,
    fill=True,
    color='red',
    label='Rejeitado'
)

# Marca o instante do evento GW150914.
ax1.axvline(tempo_evento, color='darkred', linestyle='--',
            linewidth=2, label='GW150914')

ax1.set_title('Qualidade dos dados — DATA e BURST_CAT3')
ax1.set_xlabel('Tempo desde o início do arquivo (s)')
ax1.set_yticks([])
ax1.set_ylim(0, 1.1)
ax1.set_xlim(0, Nseg)
ax1.legend(loc='upper right')

# Ajusta os espaçamentos e exibe os gráficos.
plt.tight_layout()
plt.show()


## 4.9 Relacionando a amplitude do strain com a qualidade dos dados

Para concluir a análise, vamos comparar a amplitude RMS do *strain* com os intervalos que satisfazem os critérios de qualidade `DATA` e `BURST_CAT3`.

No gráfico superior, cada ponto representa o RMS do sinal calculado durante um segundo de observação.

No gráfico inferior, as regiões azuis representam os intervalos aceitos, enquanto as regiões vermelhas indicam os intervalos rejeitados.

A linha vertical tracejada marca o instante do evento GW150914.

> **Observe:** Os maiores valores de RMS coincidem necessariamente com os intervalos rejeitados? O evento GW150914 está localizado em um intervalo que satisfaz os critérios de qualidade selecionados?
>
> **Conclusão:** A amplitude do *strain* e as flags de qualidade fornecem informações complementares. As flags ajudam a identificar intervalos adequados para diferentes análises, enquanto o RMS permite examinar variações na amplitude das flutuações do sinal.


# 5. Verificação e exercícios

**Questão 1.** No arquivo do GWOSC, o *strain* é amostrado a 4096 Hz e as flags de qualidade a 1 Hz.

**a)** Quantas amostras de *strain* correspondem a um segundo?  
**b)** Quantos valores existem em cada flag de qualidade durante 4096 segundos?


**Questão 2.** Indique o que significam os valores **0** e **1** de uma flag. Em seguida, responda: um intervalo com `DATA = 1` está necessariamente livre de *glitches*?


**Questão 3.** Observe os números inteiros presentes na máscara `DQmask`.

**a)** Escreva **31** e **127** em binário usando sete bits.  
**b)** Para cada número, informe o valor de `DATA` (bit 0) e `BURST_CAT3` (bit 6).

A célula permite conferir a conversão depois de responder.

**Questão 4.** Considere o valor `DQmask = 31`. Qual será o resultado das operações abaixo? Justifique usando os bits 0 e 6.

```python
data = (31 >> 0) & 1
burst_cat3 = (31 >> 6) & 1
```

**Questão 5.** Suponha que, durante quatro segundos consecutivos, os arrays sejam:

`data_mask = [1, 1, 1, 1]`  
`burst_cat3_mask = [0, 1, 0, 1]`

Qual será `goodData_mask_1hz = data_mask & burst_cat3_mask`? Quais segundos serão aceitos? Considere índices começando em zero.

**Questão 6.** Um segmento da lista `segList` é `(1126256645, 1126256652)`.

**a)** Qual é sua duração?  
**b)** O limite final está incluído?  
**c)** O segmento está automaticamente livre de todo ruído instrumental?

**Questão 7.** No gráfico que destaca em **vermelho** os intervalos rejeitados e em **azul** os intervalos aceitos:

**a)** O que significa uma região vermelha?  
**b)** A máscara altera os valores do *strain*?  
**c)** É possível haver um trecho rejeitado sem um pico de amplitude evidente?


**Questão 8.** O evento GW150914 ocorreu aproximadamente em GPS **1126259462,4**, e o arquivo começa em GPS **1126256640**.

**a)** Quantos segundos após o início do arquivo ocorreu o evento?  
**b)** Por que o evento pode não aparecer como um pico destacado no gráfico RMS de 1 segundo?



# 6. Qual é o próximo passo?

O [próximo passo](<04>) apresentará alguns conceitos importantes para trabalhar com dados de ondas gravitacionais.